In [ ]:
# === CELL 1: Install dependencies (HuggingFace JupyterLab) ===
# NOTE: We do NOT install vllm. Unsloth's fast_inference=True uses vLLM
# internally and crashes on HF JupyterLab with CUDA allocator / torch.compile
# errors. Instead we use fast_inference=False which uses standard HF generate().
# This is ~10-20x slower for generation but 100% stable on A100.
!pip install --upgrade pip
!pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip uninstall torchcodec -y 2>/dev/null
!pip uninstall sentence-transformers -y 2>/dev/null
!pip install openenv-core -q

## RESTART KERNEL NOW

Go to **Kernel → Restart Kernel**.

Then **skip to Cell 3** below. Do NOT re-run Cell 1.

In [ ]:
# === CELL 3: Setup after restart ===
import os, sys, gc

# ──────────────────────────────────────────────────────────────
# NO vLLM — all vLLM env vars removed.
# We use fast_inference=False so Unsloth never touches vLLM.
# TRL's GRPOTrainer uses standard model.generate() for completions.
# ──────────────────────────────────────────────────────────────

# Clear any stale GPU state from prior kernel runs
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    gc.collect()

assert torch.cuda.is_available(), "No GPU!"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"VRAM used before model load: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# Unzip and install environment
os.chdir("/data")
os.system("unzip -oq undercover_agent_city.zip")
os.system("pip install -e /data/undercover_agent_city/ -q 2>/dev/null")
sys.path.insert(0, "/data")

# Verify environment works
from undercover_agent_city.models import CityAction
from undercover_agent_city.server.city_environment import CityEnvironment
env = CityEnvironment()
obs = env.reset(task="tutorial_persona")
print(f"Env OK! {len(obs.available_actions)} actions, nearby: {[n['name'] for n in obs.nearby_npcs]}")
del env, obs

In [ ]:
# === CELL 4: Generate THREE datasets ===
import json, random, re
from datasets import Dataset

def collect_observations(task, n_episodes=200, seed_start=0):
    """Collect observations by running random episodes."""
    env = CityEnvironment()
    raw = []
    for ep in range(n_episodes):
        obs = env.reset(task=task, seed=seed_start + ep)
        rng = random.Random(seed_start + ep + 1000)
        for step in range(obs.max_turns):
            if obs.done:
                break
            raw.append({
                "prompt_text": obs.message,
                "num_actions": len(obs.available_actions),
                "actions_json": json.dumps(obs.available_actions),
                "task": task,
                "seed": seed_start + ep,
                "step_idx": step,
            })
            obs = env.step(CityAction(action=rng.randint(1, len(obs.available_actions))))
    return raw

print("Collecting tutorial_persona observations...")
raw_tutorial = collect_observations("tutorial_persona", n_episodes=200, seed_start=0)
print(f"  tutorial_persona: {len(raw_tutorial)} observations")

print("Collecting first_contact observations...")
raw_first_contact = collect_observations("first_contact", n_episodes=200, seed_start=1000)
print(f"  first_contact: {len(raw_first_contact)} observations")

print("Collecting earn_trust observations...")
raw_earn_trust = collect_observations("earn_trust", n_episodes=200, seed_start=2000)
print(f"  earn_trust: {len(raw_earn_trust)} observations")

print(f"\nTotal: {len(raw_tutorial) + len(raw_first_contact) + len(raw_earn_trust)} observations across 3 tasks")

In [ ]:
# === CELL 5: Format datasets + Define system prompt ===

# NOTE on /no_think: We put it in the system prompt so the model knows not to
# reason internally. But the CRITICAL fix is passing enable_thinking=False via
# chat_template_kwargs in GRPOConfig -- otherwise the Qwen3 chat template
# prepends <think>\n to the generation, the model fills it with thinking tokens,
# those get truncated by max_completion_length, mask_truncated_completions=True
# masks them out, and ALL generations end up identical => reward_std = 0.

SYSTEM_PROMPT = (
    "You are an undercover agent infiltrating a criminal gang in a city. /no_think\n"
    "Pick the BEST action number.\n"
    "TALK to NPCs to build trust — that is your primary objective.\n"
    "Each person responds best to a specific approach. Learn which works for whom.\n"
    "Keep your cover story CONSISTENT — never contradict what you told people.\n"
    "Output ONLY a single number. Nothing else."
)

def format_row(obs_list):
    """Convert raw observations into HF Dataset with chat-format prompts."""
    return Dataset.from_list([{
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": o["prompt_text"]},
        ],
        **{k: o[k] for k in ["num_actions", "actions_json", "task", "seed", "step_idx"]},
    } for o in obs_list])

tutorial_dataset = format_row(raw_tutorial)
first_contact_dataset = format_row(raw_first_contact)
earn_trust_dataset = format_row(raw_earn_trust)

print(f"Tutorial dataset:       {len(tutorial_dataset)} rows")
print(f"First contact dataset:  {len(first_contact_dataset)} rows")
print(f"Earn trust dataset:     {len(earn_trust_dataset)} rows")

In [ ]:
# === CELL 6: Load model ===
from unsloth import FastLanguageModel
import torch

gpu_name = torch.cuda.get_device_name(0)
MODEL_NAME = "unsloth/Qwen3-4B"
NUM_GENERATIONS = 8  # A100 80GB can handle 8 — more = better GRPO advantage estimates

max_seq_length = 1024  # Increased — observations are longer with faction+trust info
lora_rank = 32

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=False,
    max_lora_rank=lora_rank,
)
model = FastLanguageModel.get_peft_model(
    model, r=lora_rank,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=lora_rank * 2,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print(f"Model: {MODEL_NAME} | GPU: {gpu_name} | Generations: {NUM_GENERATIONS}")
print(f"LoRA: r={lora_rank}, alpha={lora_rank*2}")
print(f"max_seq_length: {max_seq_length}")
print(f"VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# === CELL 7: Define reward functions ===
import random as stdlib_random

def parse_action_number(text, max_actions):
    """Extract action number from model output. Strips any XML tags like <think>."""
    text = re.sub(r"<[^>]+>.*?</[^>]+>", "", text, flags=re.DOTALL).strip()
    text = re.sub(r"<[^>]+>", "", text).strip()
    digits = "".join(c for c in text if c.isdigit())
    if digits:
        n = int(digits[0])
        if 1 <= n <= max_actions:
            return n
    return None

def format_reward(completions, **kwargs):
    """1.0 for valid action number, -1.0 otherwise."""
    na = kwargs.get("num_actions", [8] * len(completions))
    return [
        1.0 if parse_action_number(
            c[0]["content"] if isinstance(c, list) else c, int(na[i])
        ) is not None else -1.0
        for i, c in enumerate(completions)
    ]

def env_reward(completions, num_actions, actions_json, task, seed, step_idx, **kwargs):
    """Replay environment to the exact step, execute chosen action, return scaled reward."""
    scores = []
    for i, c in enumerate(completions):
        text = c[0]["content"] if isinstance(c, list) else c
        a = parse_action_number(text, int(num_actions[i]))
        if a is None:
            scores.append(-2.0)
            continue
        try:
            e = CityEnvironment()
            o = e.reset(task=str(task[i]), seed=int(seed[i]))
            t = int(step_idx[i])
            rng = stdlib_random.Random(int(seed[i]) + 1000)
            for _ in range(t):
                if o.done:
                    break
                o = e.step(CityAction(action=rng.randint(1, len(o.available_actions))))
            if o.done:
                scores.append(-1.0)
                continue
            r = e.step(CityAction(action=a))
            scores.append((r.reward or 0.0) * 8.0)
        except Exception:
            scores.append(-2.0)
    return scores

# Quick sanity test
e = CityEnvironment()
o = e.reset(task="tutorial_persona", seed=0)
print("Reward sanity test (tutorial_persona seed=0):")
for a in o.available_actions:
    e2 = CityEnvironment()
    e2.reset(task="tutorial_persona", seed=0)
    r = e2.step(CityAction(action=a["index"]))
    print(f"  {a['description'][:50]}: reward={r.reward:.3f} (x8={r.reward*8:.3f})")
print("Reward functions OK!")

In [ ]:
# === CELL 8: Evaluation functions + Pre-training baseline ===

def evaluate_model(model, tokenizer, task, n_episodes=30, seed_start=10000):
    env = CityEnvironment()
    correct = total_talks = 0
    grades = []
    cover_blown = 0
    for ep in range(n_episodes):
        obs = env.reset(task=task, seed=seed_start + ep)
        for _ in range(obs.max_turns):
            if obs.done:
                break
            msgs = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": obs.message},
            ]
            prompt = tokenizer.apply_chat_template(
                msgs, add_generation_prompt=True, tokenize=False,
                enable_thinking=False,
            )
            inp = tokenizer(prompt, return_tensors="pt").to(model.device)
            with torch.no_grad():
                out = model.generate(
                    **inp, max_new_tokens=8,
                    temperature=0.1, do_sample=True,
                )
            resp = tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True)
            action = parse_action_number(resp, len(obs.available_actions)) or 1
            obs = env.step(CityAction(action=action))
            if obs.metadata and obs.metadata.get("was_talk_action"):
                total_talks += 1
                if obs.metadata.get("persona_correct"):
                    correct += 1
            if obs.metadata and obs.metadata.get("cover_blown"):
                cover_blown += 1
        grades.append(env.grade_episode())
    return {
        "persona_acc": correct / max(1, total_talks),
        "grade": sum(grades) / len(grades),
        "cover_intact": 1 - (cover_blown / n_episodes),
        "n_episodes": n_episodes,
    }

def evaluate_random(task, n_episodes=30, seed_start=10000):
    env = CityEnvironment()
    correct = total_talks = 0
    grades = []
    cover_blown = 0
    for ep in range(n_episodes):
        obs = env.reset(task=task, seed=seed_start + ep)
        rng = stdlib_random.Random(seed_start + ep + 5000)
        for _ in range(obs.max_turns):
            if obs.done:
                break
            obs = env.step(CityAction(action=rng.randint(1, len(obs.available_actions))))
            if obs.metadata and obs.metadata.get("was_talk_action"):
                total_talks += 1
                if obs.metadata.get("persona_correct"):
                    correct += 1
            if obs.metadata and obs.metadata.get("cover_blown"):
                cover_blown += 1
        grades.append(env.grade_episode())
    return {
        "persona_acc": correct / max(1, total_talks),
        "grade": sum(grades) / len(grades),
        "cover_intact": 1 - (cover_blown / n_episodes),
        "n_episodes": n_episodes,
    }

print("=" * 60)
print("BASELINE EVALUATION (before training)")
print("=" * 60)

baselines = {}
EVAL_TASKS = [("tutorial_persona", 30), ("first_contact", 10), ("earn_trust", 5)]

print("\nCollecting random baselines...")
baselines["tutorial_persona_random"] = evaluate_random("tutorial_persona", 30)
baselines["first_contact_random"] = evaluate_random("first_contact", 10)
baselines["earn_trust_random"] = evaluate_random("earn_trust", 5)

print("Evaluating untrained model...")
baselines["tutorial_persona_untrained"] = evaluate_model(model, tokenizer, "tutorial_persona", 30)
baselines["first_contact_untrained"] = evaluate_model(model, tokenizer, "first_contact", 10)
baselines["earn_trust_untrained"] = evaluate_model(model, tokenizer, "earn_trust", 5)

for task, n in EVAL_TASKS:
    print(f"  {task}: random persona={baselines[f'{task}_random']['persona_acc']:.0%}  grade={baselines[f'{task}_random']['grade']:.3f}")
    print(f"  {task}: untrained persona={baselines[f'{task}_untrained']['persona_acc']:.0%}  grade={baselines[f'{task}_untrained']['grade']:.3f}")

print("\nBaseline collection complete.")

In [ ]:
# === CELL 9: SINGLE MIXED TRAINING (1000 steps) ===
from trl import GRPOConfig, GRPOTrainer
from datasets import concatenate_datasets

# Build mixed dataset: 30% tutorial + 40% first_contact + 30% earn_trust
total = len(tutorial_dataset) + len(first_contact_dataset) + len(earn_trust_dataset)
target_tutorial = int(total * 0.30)
target_first = int(total * 0.40)
target_earn = total - target_tutorial - target_first

tutorial_sampled = tutorial_dataset.shuffle(seed=42).select(range(min(target_tutorial, len(tutorial_dataset))))
first_sampled = first_contact_dataset.shuffle(seed=42).select(range(min(target_first, len(first_contact_dataset))))
earn_sampled = earn_trust_dataset.shuffle(seed=42).select(range(min(target_earn, len(earn_trust_dataset))))

mixed_dataset = concatenate_datasets([tutorial_sampled, first_sampled, earn_sampled]).shuffle(seed=42)

print(f"Mixed dataset: {len(mixed_dataset)} rows")
print(f"  tutorial: {len(tutorial_sampled)} ({len(tutorial_sampled)/len(mixed_dataset):.0%})")
print(f"  first_contact: {len(first_sampled)} ({len(first_sampled)/len(mixed_dataset):.0%})")
print(f"  earn_trust: {len(earn_sampled)} ({len(earn_sampled)/len(mixed_dataset):.0%})")

# Use generation_kwargs for compatibility with older Unsloth/TRL versions
# that don't support chat_template_kwargs directly.
# enable_thinking=False prevents Qwen3 from generating <think> tokens
# that consume max_completion_length and cause reward_std=0.
grpo_config = GRPOConfig(
    temperature=1.0,
    top_p=0.95,
    num_generations=NUM_GENERATIONS,
    max_prompt_length=900,
    max_completion_length=32,
    generation_kwargs={"enable_thinking": False},
    learning_rate=5e-6,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    logging_steps=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=1000,
    save_steps=1000,
    max_grad_norm=0.1,
    epsilon=0.2,
    epsilon_high=0.28,
    mask_truncated_completions=True,
    report_to="none",
    output_dir="/data/outputs_mixed",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
)

print("=" * 60)
print("MIXED TRAINING: 1000 steps")
print(f"  generation_kwargs: {grpo_config.generation_kwargs}")
print(f"  max_completion_length: {grpo_config.max_completion_length}")
print("=" * 60)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[format_reward, env_reward],
    reward_weights=[1.0, 8.0],
    args=grpo_config,
    train_dataset=mixed_dataset,
)
trainer.train()
train_logs = trainer.state.log_history.copy()
print("Training complete!")

In [ ]:
# === CELL 10: Evaluate AFTER training ===
print("=" * 60)
print("POST-TRAINING EVALUATION")
print("=" * 60)

eval_trained = {}
eval_trained["tutorial_persona"] = evaluate_model(model, tokenizer, "tutorial_persona", 30)
eval_trained["first_contact"] = evaluate_model(model, tokenizer, "first_contact", 10)
eval_trained["earn_trust"] = evaluate_model(model, tokenizer, "earn_trust", 5)

for task in ["tutorial_persona", "first_contact", "earn_trust"]:
    print(f"  {task}: persona={eval_trained[task]['persona_acc']:.0%}  grade={eval_trained[task]['grade']:.3f}")

print("\n" + "=" * 80)
print("BEFORE / AFTER COMPARISON")
print("=" * 80)
print(f"{'Task':<20} {'Metric':<10} {'Random':>8} {'Untrained':>10} {'Trained':>8} {'Delta':>8}")
print("-" * 80)
for task, _ in EVAL_TASKS:
    rp = baselines[f"{task}_random"]["persona_acc"]
    up = baselines[f"{task}_untrained"]["persona_acc"]
    tp = eval_trained[task]["persona_acc"]
    print(f"{task:<20} {'Persona':<10} {rp:>7.0%} {up:>9.0%} {tp:>7.0%} {tp-up:>+7.0%}")
    rg = baselines[f"{task}_random"]["grade"]
    ug = baselines[f"{task}_untrained"]["grade"]
    tg = eval_trained[task]["grade"]
    print(f"{'':<20} {'Grade':<10} {rg:>8.3f} {ug:>10.3f} {tg:>8.3f} {tg-ug:>+8.3f}")
    print()

In [ ]:
# === CELL 11: Generate plots ===
import matplotlib.pyplot as plt
import numpy as np
import os
os.makedirs("/data/outputs/plots", exist_ok=True)

# --- Plot 1: 1000-step reward curve ---
steps, rewards = [], []
for entry in train_logs:
    if "reward" in entry:
        steps.append(entry["step"])
        rewards.append(entry["reward"])

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(steps, rewards, alpha=0.3, lw=0.5, color="green")
if len(rewards) > 20:
    w = 20
    smoothed = np.convolve(rewards, np.ones(w) / w, mode="valid")
    ax.plot(steps[:len(smoothed)], smoothed, color="green", lw=2, label="Smoothed reward")
ax.set(xlabel="Training Step", ylabel="Reward",
       title="Undercover Agent City -- GRPO Mixed Training (1000 steps)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("/data/outputs/plots/reward_curve.png", dpi=150)
plt.show()
print("Saved: /data/outputs/plots/reward_curve.png")

# --- Plot 2: Before/after bar charts ---
tasks = ["tutorial_persona", "first_contact", "earn_trust"]
x = np.arange(len(tasks))
width = 0.2

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, (label, data) in enumerate([
    ("Random", {t: baselines[f"{t}_random"]["persona_acc"] for t in tasks}),
    ("Untrained", {t: baselines[f"{t}_untrained"]["persona_acc"] for t in tasks}),
    ("Trained", {t: eval_trained[t]["persona_acc"] for t in tasks}),
]):
    axes[0].bar(x + i * width, [data[t] for t in tasks], width, label=label)
axes[0].set(ylabel="Persona Accuracy", title="Persona Accuracy: Before vs After")
axes[0].set_xticks(x + width)
axes[0].set_xticklabels(tasks, rotation=15)
axes[0].legend(fontsize=9)
axes[0].set_ylim(0, 1.1)

for i, (label, data) in enumerate([
    ("Random", {t: baselines[f"{t}_random"]["grade"] for t in tasks}),
    ("Untrained", {t: baselines[f"{t}_untrained"]["grade"] for t in tasks}),
    ("Trained", {t: eval_trained[t]["grade"] for t in tasks}),
]):
    axes[1].bar(x + i * width, [data[t] for t in tasks], width, label=label)
axes[1].set(ylabel="Grade (0-1)", title="Task Grade: Before vs After")
axes[1].set_xticks(x + width)
axes[1].set_xticklabels(tasks, rotation=15)
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 1.1)

plt.suptitle("Mixed Training Results (1000 steps, A100)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/data/outputs/plots/eval_comparison.png", dpi=150)
plt.show()
print("Saved: /data/outputs/plots/eval_comparison.png")

In [ ]:
# === CELL 12: Save LoRA adapter + results ===
import shutil, json

# With fast_inference=False, use standard PEFT save (not model.save_lora)
LORA_DIR = "/data/undercover_agent_city_lora"
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)

os.makedirs("/data/outputs/model", exist_ok=True)
if os.path.exists(LORA_DIR):
    shutil.copytree(LORA_DIR, "/data/outputs/model/lora_adapter", dirs_exist_ok=True)

results = {
    "model": MODEL_NAME,
    "lora_rank": 32,
    "lora_alpha": 64,
    "total_steps": 1000,
    "fast_inference": False,
    "vllm": False,
    "training": "mixed (30% tutorial + 40% first_contact + 30% earn_trust)",
    "baselines": baselines,
    "eval_trained": eval_trained,
}
with open("/data/outputs/results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

!cd /data && zip -rq outputs.zip outputs/

print("\nDone! Artifacts saved to /data/outputs/")
print("  /data/outputs/plots/reward_curve.png")
print("  /data/outputs/plots/eval_comparison.png")
print("  /data/outputs/model/lora_adapter/")
print("  /data/outputs/results.json")
print("  /data/outputs.zip (download this)")